# 🍄 End-to-End Smart Packaging Intelligence for White Button Mushrooms
## Integrated Pipeline: Biological Spoilage Classification + Multimodal Remaining Shelf Life (RSL) Prediction
**Complete Research Pipeline:**
1. **Part I (Quality Grading & Explainability):** 4-Class physiological quality grading + Grad-CAM attention mapping.
2. **Domain-Specific Weight Transfer:** Reuses Part I mushroom defect representations into Part II.
3. **Part II (Multimodal RSL Regression):** Deep feature fusion (Mushroom Visuals + Essential Oil Packaging Formulation) to predict continuous shelf life (days).
4. **Rigorous Evaluation:** Zero data-leakage (Sample-ID split) across both phases.

In [ ]:
import os
import sys
import time
import zipfile
import shutil
from pathlib import Path

import torch
import torch.nn as nn
from torchvision import transforms
import numpy as np
import cv2
import matplotlib.pyplot as plt
from PIL import Image

print('=' * 70)
print('ENVIRONMENT & HARDWARE ACCELERATION CHECK')
print('=' * 70)
print(f'PyTorch Version:    {torch.__version__}')
print(f'CUDA Available:     {torch.cuda.is_available()}')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Active Device:      {device}')
if device.type == 'cuda':
    print(f'GPU Device Name:    {torch.cuda.get_device_name(0)}')
    print(f'GPU Memory Total:   {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB')
print('=' * 70)


### 1. Dataset Extraction & Directory Resolution
Extracts the preprocessed 12.9 MB dataset archive (mushroom_preprocessed_224.zip) into /kaggle/working/dataset/.


In [ ]:
DATASET_DIR = Path('/kaggle/working/dataset')
DATASET_DIR.mkdir(parents=True, exist_ok=True)

# 1. Search for zip in Kaggle input or local directory
zip_candidates = list(Path('/kaggle/input').glob('**/*mushroom*.zip')) + list(Path('.').glob('**/*mushroom*.zip'))

if zip_candidates:
    chosen_zip = zip_candidates[0]
    print(f'Found dataset archive: {chosen_zip}')
    with zipfile.ZipFile(chosen_zip, 'r') as zf:
        zf.extractall(DATASET_DIR)
    print(f'Extracted dataset to: {DATASET_DIR}')
else:
    # Check if files already present in input
    pre_candidates = list(Path('/kaggle/input').glob('**/Control'))
    if pre_candidates:
        DATASET_DIR = pre_candidates[0].parent
        print(f'Using pre-mounted dataset directory: {DATASET_DIR}')
    else:
        print('[Warning] No zip found in /kaggle/input. Using current directory.')
        DATASET_DIR = Path('preprocessed_224').resolve()

# Verify contents
treatments = [d.name for d in DATASET_DIR.iterdir() if d.is_dir() and not d.name.startswith('.')]
print(f'Discovered treatments: {treatments}')
image_count = len(list(DATASET_DIR.glob('*/*/*.*')))
print(f'Total preprocessed images available: {image_count} (Expected: 792)')


---
# 🔬 PART I: Biological Quality Grade Classification & Grad-CAM
Dynamically maps (Treatment, Day) to 4 physiological grades accounting for active packaging preservation.
Trains ResNet-50 vs. Swin-T and saves optimal feature representation weights to `checkpoints/`.

### 2. Biological Quality Grade Labeling & PyTorch Dataset
Maps (Treatment, Day) to 4 physiological quality grades to account for packaging spoilage delays.
Ensures zero data leakage by strictly partitioning by **Sample ID (1 to 22)**.


In [ ]:
LABEL_MAPPING = {
    # Class 0 (Very Healthy): Control Day 0 | 1SKEO.2CEO Days 0, 3 | 2SKEO.1CEO Days 0, 3
    ('Control', 0): 0, ('1SKEO.2CEO', 0): 0, ('1SKEO.2CEO', 3): 0,
    ('2SKEO.1CEO', 0): 0, ('2SKEO.1CEO', 3): 0,

    # Class 1 (Healthy): Control Day 3 | 1SKEO.2CEO Days 7, 11 | 2SKEO.1CEO Days 7, 11
    ('Control', 3): 1, ('1SKEO.2CEO', 7): 1, ('1SKEO.2CEO', 11): 1,
    ('2SKEO.1CEO', 7): 1, ('2SKEO.1CEO', 11): 1,

    # Class 2 (Spoiled): Control Days 7, 11 | 1SKEO.2CEO Day 20 | 2SKEO.1CEO Day 20
    ('Control', 7): 2, ('Control', 11): 2,
    ('1SKEO.2CEO', 20): 2, ('2SKEO.1CEO', 20): 2,

    # Class 3 (Very Spoiled): Control Days 20, 30 | 1SKEO.2CEO Day 30 | 2SKEO.1CEO Day 30
    ('Control', 20): 3, ('Control', 30): 3,
    ('1SKEO.2CEO', 30): 3, ('2SKEO.1CEO', 30): 3,
}

CLASS_NAMES = [
    'Class 0: Very Healthy',
    'Class 1: Healthy',
    'Class 2: Spoiled',
    'Class 3: Very Spoiled'
]

import json, re

def get_splits(data_dir):
    split_file = data_dir / 'splits.json'
    if split_file.exists():
        with open(split_file, 'r') as f:
            return json.load(f)
    # Default deterministic partition (seed=42)
    return {
        'train': [2, 3, 5, 6, 7, 10, 11, 12, 13, 14, 15, 16, 17, 19, 20, 22],
        'val': [8, 9, 18],
        'test': [1, 4, 21]
    }

class MushroomDataset(torch.utils.data.Dataset):
    def __init__(self, root_dir, split='train', transform=None):
        self.root_dir = Path(root_dir)
        self.split = split.lower()
        self.transform = transform
        self.splits = get_splits(self.root_dir)
        self.allowed_ids = set(self.splits[self.split]) if self.split != 'all' else set(range(1, 23))
        self.samples = self._load_samples()

    def _load_samples(self):
        samples = []
        pattern = re.compile(r'^(\d+)-([bBsS])\.')
        for img_path in sorted(self.root_dir.glob('*/*/*.*')):
            if img_path.suffix.lower() not in ('.jpg', '.jpeg', '.png'):
                continue
            t_name = img_path.parent.parent.name
            day = int(img_path.parent.name) if img_path.parent.name.isdigit() else -1
            if (t_name, day) not in LABEL_MAPPING:
                continue
            m = pattern.match(img_path.name)
            if not m:
                continue
            s_id = int(m.group(1))
            if s_id in self.allowed_ids:
                samples.append({
                    'path': str(img_path),
                    'treatment': t_name,
                    'day': day,
                    'sample_id': s_id,
                    'label': LABEL_MAPPING[(t_name, day)]
                })
        return samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        item = self.samples[idx]
        image = Image.open(item['path']).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, item['label']

# Transforms & Loaders
train_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

eval_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_ds = MushroomDataset(DATASET_DIR, split='train', transform=train_tf)
val_ds = MushroomDataset(DATASET_DIR, split='val', transform=eval_tf)
test_ds = MushroomDataset(DATASET_DIR, split='test', transform=eval_tf)

train_loader = torch.utils.data.DataLoader(train_ds, batch_size=16, shuffle=True, num_workers=2, pin_memory=True)
val_loader = torch.utils.data.DataLoader(val_ds, batch_size=16, shuffle=False, num_workers=2, pin_memory=True)
test_loader = torch.utils.data.DataLoader(test_ds, batch_size=16, shuffle=False, num_workers=2, pin_memory=True)

print(f'Train Samples: {len(train_ds)} (16 IDs) | Val Samples: {len(val_ds)} (3 IDs) | Test Samples: {len(test_ds)} (3 IDs)')


### 3. Model Building & Transfer Learning (ResNet50 vs. Swin-T)
Both pre-trained architectures adapted for 4 Visual Quality Classes with Two-Stage Fine-Tuning support.


In [ ]:
from torchvision.models import resnet50, ResNet50_Weights, swin_t, Swin_T_Weights

def build_resnet50(num_classes=4, pretrained=True):
    weights = ResNet50_Weights.IMAGENET1K_V2 if pretrained else None
    model = resnet50(weights=weights)
    model.fc = nn.Sequential(
        nn.Dropout(p=0.3),
        nn.Linear(model.fc.in_features, num_classes)
    )
    model.model_name = 'resnet50'
    return model

def build_swin_t(num_classes=4, pretrained=True):
    weights = Swin_T_Weights.IMAGENET1K_V1 if pretrained else None
    model = swin_t(weights=weights)
    model.head = nn.Sequential(
        nn.Dropout(p=0.3),
        nn.Linear(model.head.in_features, num_classes)
    )
    model.model_name = 'swin_t'
    return model

def freeze_backbone(model):
    is_resnet = 'resnet' in model.model_name
    for name, param in model.named_parameters():
        if ('fc' if is_resnet else 'head') in name:
            param.requires_grad = True
        else:
            param.requires_grad = False

def unfreeze_backbone(model):
    for param in model.parameters():
        param.requires_grad = True

def get_optimizer_groups(model, backbone_lr=1e-4, head_lr=1e-3, weight_decay=1e-4):
    head_names = {'fc', 'head'}
    backbone_params = []
    head_params = []
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        if any(h in name for h in head_names):
            head_params.append(param)
        else:
            backbone_params.append(param)
    return [
        {'params': backbone_params, 'lr': backbone_lr, 'weight_decay': weight_decay},
        {'params': head_params, 'lr': head_lr, 'weight_decay': weight_decay}
    ]

print('Model builders and fine-tuning utilities successfully initialized.')


### 4. Training Engine with Two-Stage Fine-Tuning & Early Stopping


In [ ]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

def train_model(model_name, total_epochs=35, warmup_epochs=5, patience=8):
    print('=' * 75)
    print(f'STARTING TWO-STAGE TRAINING: {model_name.upper()}')
    print('=' * 75)

    model = (build_resnet50() if model_name == 'resnet50' else build_swin_t()).to(device)
    criterion = nn.CrossEntropyLoss()
    os.makedirs('checkpoints', exist_ok=True)
    best_ckpt = f'checkpoints/best_{model_name}.pth'

    # Stage 1: Warmup Classifier Head
    if warmup_epochs > 0:
        print(f'>>> Stage 1: Classifier Head Warmup ({warmup_epochs} epochs, frozen backbone) <<<')
        freeze_backbone(model)
        optimizer_warmup = AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-3, weight_decay=1e-4)
        for ep in range(1, warmup_epochs + 1):
            model.train()
            t_loss, t_corr, t_tot = 0.0, 0, 0
            for imgs, targets in train_loader:
                imgs, targets = imgs.to(device), targets.to(device)
                optimizer_warmup.zero_grad()
                outs = model(imgs)
                loss = criterion(outs, targets)
                loss.backward()
                optimizer_warmup.step()
                t_loss += loss.item() * targets.size(0)
                t_corr += outs.argmax(1).eq(targets).sum().item()
                t_tot += targets.size(0)

            model.eval()
            v_loss, v_corr, v_tot = 0.0, 0, 0
            with torch.no_grad():
                for imgs, targets in val_loader:
                    imgs, targets = imgs.to(device), targets.to(device)
                    outs = model(imgs)
                    v_loss += criterion(outs, targets).item() * targets.size(0)
                    v_corr += outs.argmax(1).eq(targets).sum().item()
                    v_tot += targets.size(0)

            print(f'  [Warmup {ep:02d}/{warmup_epochs:02d}] Train Loss: {t_loss/t_tot:.4f}, Acc: {t_corr/t_tot*100:.1f}% || Val Loss: {v_loss/v_tot:.4f}, Acc: {v_corr/v_tot*100:.1f}%')

    # Stage 2: End-to-End Fine-Tuning
    print(f'>>> Stage 2: End-to-End Fine-Tuning ({total_epochs - warmup_epochs} epochs) <<<')
    unfreeze_backbone(model)
    optimizer = AdamW(get_optimizer_groups(model, backbone_lr=1e-4, head_lr=1e-3))
    scheduler = CosineAnnealingLR(optimizer, T_max=total_epochs - warmup_epochs, eta_min=1e-6)

    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    best_val_loss = float('inf')
    best_val_acc = 0.0
    patience_cnt = 0

    for ep in range(1, total_epochs - warmup_epochs + 1):
        cur_ep = warmup_epochs + ep
        model.train()
        t_loss, t_corr, t_tot = 0.0, 0, 0
        for imgs, targets in train_loader:
            imgs, targets = imgs.to(device), targets.to(device)
            optimizer.zero_grad()
            outs = model(imgs)
            loss = criterion(outs, targets)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            t_loss += loss.item() * targets.size(0)
            t_corr += outs.argmax(1).eq(targets).sum().item()
            t_tot += targets.size(0)
        scheduler.step()

        model.eval()
        v_loss, v_corr, v_tot = 0.0, 0, 0
        with torch.no_grad():
            for imgs, targets in val_loader:
                imgs, targets = imgs.to(device), targets.to(device)
                outs = model(imgs)
                v_loss += criterion(outs, targets).item() * targets.size(0)
                v_corr += outs.argmax(1).eq(targets).sum().item()
                v_tot += targets.size(0)

        tr_l, tr_a = t_loss / t_tot, (t_corr / t_tot) * 100.0
        va_l, va_a = v_loss / v_tot, (v_corr / v_tot) * 100.0
        history['train_loss'].append(tr_l)
        history['val_loss'].append(va_l)
        history['train_acc'].append(tr_a)
        history['val_acc'].append(va_a)

        mark = ''
        if va_l < best_val_loss:
            best_val_loss = va_l
            best_val_acc = va_a
            patience_cnt = 0
            torch.save({'state_dict': model.state_dict(), 'val_acc': best_val_acc}, best_ckpt)
            mark = ' [* BEST]'
        else:
            patience_cnt += 1

        print(f'  [Epoch {cur_ep:02d}/{total_epochs:02d}] Train Loss: {tr_l:.4f}, Acc: {tr_a:.1f}% || Val Loss: {va_l:.4f}, Acc: {va_a:.1f}%{mark}')
        if patience_cnt >= patience:
            print(f'Early stopping triggered after {patience} epochs without improvement.')
            break

    print(f'Finished {model_name}. Best Val Acc: {best_val_acc:.2f}% | Checkpoint saved: {best_ckpt}')
    return history


### 5. Training Execution: ResNet50 & Swin Transformer


In [ ]:
# Train ResNet50
history_resnet = train_model('resnet50', total_epochs=35, warmup_epochs=5, patience=8)

# Train Swin Transformer
history_swin = train_model('swin_t', total_epochs=35, warmup_epochs=5, patience=8)

# Plot Comparative Learning Curves
plt.figure(figsize=(14, 5))
plt.subplot(1, 2, 1)
plt.plot(history_resnet['val_loss'], 'b-o', label='ResNet50 Val Loss')
plt.plot(history_swin['val_loss'], 'r-s', label='Swin-T Val Loss')
plt.title('Validation Loss Comparison', fontweight='bold')
plt.xlabel('Fine-Tuning Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True, linestyle='--')

plt.subplot(1, 2, 2)
plt.plot(history_resnet['val_acc'], 'b-o', label='ResNet50 Val Acc (%)')
plt.plot(history_swin['val_acc'], 'r-s', label='Swin-T Val Acc (%)')
plt.title('Validation Accuracy Comparison (%)', fontweight='bold')
plt.xlabel('Fine-Tuning Epoch')
plt.ylabel('Accuracy (%)')
plt.legend()
plt.grid(True, linestyle='--')

os.makedirs('outputs', exist_ok=True)
plt.tight_layout()
plt.savefig('outputs/training_curves_comparison.png', dpi=200)
plt.show()


### 6. Test Set Evaluation & Confusion Matrices (Held-Out Samples: 1, 4, 21)


In [ ]:
def evaluate_and_plot_cm(model_name):
    model = (build_resnet50(pretrained=False) if model_name == 'resnet50' else build_swin_t(pretrained=False)).to(device)
    ckpt = torch.load(f'checkpoints/best_{model_name}.pth', map_location=device)
    model.load_state_dict(ckpt['state_dict'])
    model.eval()

    y_true, y_pred = [], []
    with torch.no_grad():
        for imgs, targets in test_loader:
            imgs = imgs.to(device)
            outs = model(imgs)
            y_true.extend(targets.numpy())
            y_pred.extend(outs.argmax(1).cpu().numpy())

    y_true, y_pred = np.array(y_true), np.array(y_pred)
    cm = np.zeros((4, 4), dtype=int)
    for t, p in zip(y_true, y_pred):
        cm[t, p] += 1
    acc = np.trace(cm) / len(y_true) * 100.0

    # Plot CM
    cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(cm_norm, cmap='Blues')
    plt.colorbar(im)
    short_names = ['Very Healthy', 'Healthy', 'Spoiled', 'Very Spoiled']
    ax.set_xticks(range(4))
    ax.set_yticks(range(4))
    ax.set_xticklabels(short_names, rotation=20, ha='right')
    ax.set_yticklabels(short_names)
    ax.set_xlabel('Predicted Quality Grade', fontweight='bold')
    ax.set_ylabel('True Quality Grade', fontweight='bold')
    ax.set_title(f'{model_name.upper()} Test Set CM (Acc: {acc:.2f}%)', fontweight='bold')

    for i in range(4):
        for j in range(4):
            color = 'white' if cm_norm[i, j] > 0.5 else 'black'
            ax.text(j, i, f'{cm[i, j]}\n({cm_norm[i, j]*100:.1f}%)', ha='center', va='center', color=color, fontweight='bold')

    plt.tight_layout()
    plt.savefig(f'outputs/confusion_matrix_{model_name}.png', dpi=200)
    plt.show()
    print(f'{model_name.upper()} Test Accuracy: {acc:.2f}%')

evaluate_and_plot_cm('resnet50')
evaluate_and_plot_cm('swin_t')


### 7. Grad-CAM Explainability: Defect Focus vs. Background Glare
Generates side-by-side attention maps proving whether ResNet50 and Swin Transformer focus on actual surface defects (browning & shrinkage).


In [ ]:
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.acts = None
        self.grads = None
        target_layer.register_forward_hook(lambda m, inp, out: setattr(self, 'acts', out.detach()))
        target_layer.register_full_backward_hook(lambda m, gin, gout: setattr(self, 'grads', gout[0].detach()))

    def generate(self, input_tensor, target_class):
        self.model.eval()
        self.model.zero_grad()
        logits = self.model(input_tensor)
        score = logits[0, target_class]
        score.backward()

        acts, grads = self.acts, self.grads
        if acts.ndim == 4 and acts.shape[-1] in (768, 1024, 2048):
            acts = acts.permute(0, 3, 1, 2)
            grads = grads.permute(0, 3, 1, 2)

        w = torch.mean(grads, dim=(2, 3), keepdim=True)
        cam = torch.sum(w * acts, dim=1, keepdim=True)
        cam = torch.relu(cam).squeeze().cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam

# Initialize models and Grad-CAM
res_model = build_resnet50(pretrained=False).to(device)
res_model.load_state_dict(torch.load('checkpoints/best_resnet50.pth', map_location=device)['state_dict'])
res_cam = GradCAM(res_model, res_model.layer4[-1])

swin_model = build_swin_t(pretrained=False).to(device)
swin_model.load_state_dict(torch.load('checkpoints/best_swin_t.pth', map_location=device)['state_dict'])
swin_cam = GradCAM(swin_model, swin_model.features[7])

# Select representative test conditions
samples_to_show = [
    ('Control Day 0 (Fresh / Very Healthy)', DATASET_DIR / 'Control' / '0' / '1-b.jpg', 0),
    ('Control Day 7 (Browning Onset / Spoiled)', DATASET_DIR / 'Control' / '7' / '1-b.jpg', 2),
    ('1SKEO.2CEO Day 20 (Active Packaging Delay)', DATASET_DIR / '1SKEO.2CEO' / '20' / '1-b.jpg', 2),
    ('Control Day 20 (Severe Spoilage & Browning)', DATASET_DIR / 'Control' / '20' / '1-b.jpg', 3),
]

valid_show = [s for s in samples_to_show if Path(s[1]).exists()]
if valid_show:
    fig, axes = plt.subplots(len(valid_show), 3, figsize=(14, 4 * len(valid_show)))
    if len(valid_show) == 1:
        axes = np.expand_dims(axes, 0)

    for idx, (title, p, lbl) in enumerate(valid_show):
        img = Image.open(p).convert('RGB')
        t_in = eval_tf(img).unsqueeze(0).to(device)
        img_np = np.array(img.resize((224, 224)))

        h_res = res_cam.generate(t_in, lbl)
        h_swin = swin_cam.generate(t_in, lbl)

        hm_res_rgb = cv2.applyColorMap(np.uint8(255 * cv2.resize(h_res, (224, 224))), cv2.COLORMAP_JET)
        hm_swin_rgb = cv2.applyColorMap(np.uint8(255 * cv2.resize(h_swin, (224, 224))), cv2.COLORMAP_JET)

        ov_res = cv2.cvtColor(np.uint8(0.5 * img_np + 0.5 * cv2.cvtColor(hm_res_rgb, cv2.COLOR_BGR2RGB)), cv2.COLOR_RGB2BGR)
        ov_swin = cv2.cvtColor(np.uint8(0.5 * img_np + 0.5 * cv2.cvtColor(hm_swin_rgb, cv2.COLOR_BGR2RGB)), cv2.COLOR_RGB2BGR)

        axes[idx, 0].imshow(img_np)
        axes[idx, 0].set_title(f'{title}\nTarget: {CLASS_NAMES[lbl]}', fontweight='bold', fontsize=10)
        axes[idx, 0].axis('off')

        axes[idx, 1].imshow(cv2.cvtColor(ov_res, cv2.COLOR_BGR2RGB))
        axes[idx, 1].set_title('ResNet50 Heatmap (CNN)', fontweight='bold', fontsize=10, color='navy')
        axes[idx, 1].axis('off')

        axes[idx, 2].imshow(cv2.cvtColor(ov_swin, cv2.COLOR_BGR2RGB))
        axes[idx, 2].set_title('Swin-T Heatmap (Vision Transformer)', fontweight='bold', fontsize=10, color='darkred')
        axes[idx, 2].axis('off')

    plt.tight_layout()
    plt.savefig('outputs/gradcam_comparison.png', dpi=200)
    plt.show()
    print('Grad-CAM comparison successfully generated and saved to outputs/gradcam_comparison.png')


---
# ⏱️ PART II: Multimodal Remaining Shelf Life (RSL) Prediction
## Initialized with Part I Mushroom-Trained Backbone Weights
Fuses deep visual features with active packaging formulations to predict continuous Remaining Shelf Life (RSL in Days).

### 2. Remaining Shelf Life (RSL) Formulation & Multimodal Dataset
- **Commercial Usability Endpoints:**
  - Control (Uncoated): Spoilage onset is Day 7 $ightarrow$ Shelf life = 7 days.
  - Active Packaging (1SKEO.2CEO & 2SKEO.1CEO): Spoilage onset delayed to Day 20 $ightarrow$ Shelf life = 20 days.
- **Continuous Target:** $RSL = \max(0, 	ext{Threshold} - 	ext{Current Day})$.
- **Multimodal Encoding:** Treatment formulation is encoded as a One-Hot vector `[Control, 1SKEO.2CEO, 2SKEO.1CEO]`.
- **Advanced Augmentation:** Flips, Rotations (±25°), Affine transforms, ColorJitter, and RandomErasing.


In [ ]:
# RSL Thresholds based on biological spoilage onset (Class 2)
SPOILAGE_THRESHOLDS = {
    'Control': 7.0,
    '1SKEO.2CEO': 20.0,
    '2SKEO.1CEO': 20.0,
}

TREATMENT_TO_INDEX = {
    'Control': 0,
    '1SKEO.2CEO': 1,
    '2SKEO.1CEO': 2,
}

def calculate_rsl(treatment, current_day):
    thresh = SPOILAGE_THRESHOLDS.get(treatment, 7.0)
    return float(max(0.0, thresh - current_day))

def get_splits(data_dir):
    split_file = data_dir / 'splits.json'
    if split_file.exists():
        with open(split_file, 'r') as f:
            return json.load(f)
    return {
        'train': [2, 3, 5, 6, 7, 10, 11, 12, 13, 14, 15, 16, 17, 19, 20, 22],
        'val': [8, 9, 18],
        'test': [1, 4, 21]
    }

class MushroomRSLDataset(torch.utils.data.Dataset):
    def __init__(self, root_dir, split='train', transform=None):
        self.root_dir = Path(root_dir)
        self.split = split.lower()
        self.transform = transform
        self.splits = get_splits(self.root_dir)
        self.allowed_ids = set(self.splits[self.split]) if self.split != 'all' else set(range(1, 23))
        self.samples = self._load_samples()

    def _load_samples(self):
        samples = []
        pattern = re.compile(r'^(\d+)-([bBsS])\.')
        for img_path in sorted(self.root_dir.glob('*/*/*.*')):
            if img_path.suffix.lower() not in ('.jpg', '.jpeg', '.png'):
                continue
            t_name = img_path.parent.parent.name
            if t_name not in TREATMENT_TO_INDEX:
                continue
            day = int(img_path.parent.name) if img_path.parent.name.isdigit() else -1
            if day < 0:
                continue
            m = pattern.match(img_path.name)
            if not m:
                continue
            s_id = int(m.group(1))
            if s_id in self.allowed_ids:
                rsl = calculate_rsl(t_name, day)
                samples.append({
                    'path': str(img_path),
                    'treatment': t_name,
                    'treatment_idx': TREATMENT_TO_INDEX[t_name],
                    'day': day,
                    'sample_id': s_id,
                    'rsl': rsl
                })
        return samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        item = self.samples[idx]
        image = Image.open(item['path']).convert('RGB')
        if self.transform:
            image = self.transform(image)

        treatment_one_hot = torch.zeros(3, dtype=torch.float32)
        treatment_one_hot[item['treatment_idx']] = 1.0

        target_rsl = torch.tensor([item['rsl']], dtype=torch.float32)
        return image, treatment_one_hot, target_rsl

# Advanced Augmentation Pipeline
train_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=25),
    transforms.RandomAffine(degrees=0, translate=(0.06, 0.06), scale=(0.92, 1.08)),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.25, scale=(0.02, 0.2), value='random')
])

eval_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_ds = MushroomRSLDataset(DATASET_DIR, split='train', transform=train_tf)
val_ds = MushroomRSLDataset(DATASET_DIR, split='val', transform=eval_tf)
test_ds = MushroomRSLDataset(DATASET_DIR, split='test', transform=eval_tf)

train_loader = torch.utils.data.DataLoader(train_ds, batch_size=16, shuffle=True, num_workers=2, pin_memory=True)
val_loader = torch.utils.data.DataLoader(val_ds, batch_size=16, shuffle=False, num_workers=2, pin_memory=True)
test_loader = torch.utils.data.DataLoader(test_ds, batch_size=16, shuffle=False, num_workers=2, pin_memory=True)

print(f'Train Samples: {len(train_ds)} (16 IDs) | Val Samples: {len(val_ds)} (3 IDs) | Test Samples: {len(test_ds)} (3 IDs)')


### 3 (Part II). Multimodal Regression Architectures with Domain-Specific Transfer Learning
Initializes vision backbones with **Mushroom-Trained Weights from Part I** (`checkpoints/best_*.pth`),
giving the model immediate knowledge of enzymatic browning defects rather than generic ImageNet features.

In [ ]:
from torchvision.models import resnet50, ResNet50_Weights, swin_t, Swin_T_Weights

class MultimodalSwinRSL(nn.Module):
    def __init__(self, pretrained_phase1=True, treatment_dim=3, emb_dim=32, dropout=0.3):
        super().__init__()
        self.model_name = 'multimodal_swin_t'
        base_swin = swin_t(weights=Swin_T_Weights.IMAGENET1K_V1)

        # Transfer mushroom defect representations from Part I
        phase1_ckpt_path = Path('checkpoints/best_swin_t.pth')
        if pretrained_phase1 and phase1_ckpt_path.exists():
            print(f'>>> Transferring Part I Mushroom Weights from: {phase1_ckpt_path} <<<')
            ckpt = torch.load(phase1_ckpt_path, map_location='cpu')
            st = ckpt.get('state_dict', ckpt)
            filtered = {k: v for k, v in st.items() if not k.startswith('head.')}
            base_swin.load_state_dict(filtered, strict=False)
            print('Part I Swin-T backbone weights transferred successfully!')
        else:
            print('Using default ImageNet weights for Swin-T.')

        self.features = base_swin.features
        self.norm = base_swin.norm
        self.permute = base_swin.permute
        self.avgpool = base_swin.avgpool
        vision_dim = 768

        self.treatment_net = nn.Sequential(
            nn.Linear(treatment_dim, emb_dim),
            nn.LayerNorm(emb_dim),
            nn.GELU()
        )

        fusion_dim = vision_dim + emb_dim
        self.regression_head = nn.Sequential(
            nn.Linear(fusion_dim, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(p=dropout),
            nn.Linear(256, 64),
            nn.LayerNorm(64),
            nn.GELU(),
            nn.Dropout(p=dropout * 0.5),
            nn.Linear(64, 1),
            nn.ReLU()
        )

    def extract_visual_features(self, x):
        x = self.features(x)
        x = self.norm(x)
        x = self.permute(x)
        x = self.avgpool(x)
        return torch.flatten(x, 1)

    def forward(self, image, treatment_one_hot):
        vis = self.extract_visual_features(image)
        treat = self.treatment_net(treatment_one_hot)
        fused = torch.cat([vis, treat], dim=1)
        return self.regression_head(fused)

class MultimodalResNetRSL(nn.Module):
    def __init__(self, pretrained_phase1=True, treatment_dim=3, emb_dim=32, dropout=0.3):
        super().__init__()
        self.model_name = 'multimodal_resnet50'
        base_resnet = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)

        # Transfer mushroom defect representations from Part I
        phase1_ckpt_path = Path('checkpoints/best_resnet50.pth')
        if pretrained_phase1 and phase1_ckpt_path.exists():
            print(f'>>> Transferring Part I Mushroom Weights from: {phase1_ckpt_path} <<<')
            ckpt = torch.load(phase1_ckpt_path, map_location='cpu')
            st = ckpt.get('state_dict', ckpt)
            filtered = {k: v for k, v in st.items() if not k.startswith('fc.')}
            base_resnet.load_state_dict(filtered, strict=False)
            print('Part I ResNet50 backbone weights transferred successfully!')
        else:
            print('Using default ImageNet weights for ResNet50.')

        self.conv1 = base_resnet.conv1
        self.bn1 = base_resnet.bn1
        self.relu = base_resnet.relu
        self.maxpool = base_resnet.maxpool
        self.layer1 = base_resnet.layer1
        self.layer2 = base_resnet.layer2
        self.layer3 = base_resnet.layer3
        self.layer4 = base_resnet.layer4
        self.avgpool = base_resnet.avgpool
        vision_dim = 2048

        self.treatment_net = nn.Sequential(
            nn.Linear(treatment_dim, emb_dim),
            nn.LayerNorm(emb_dim),
            nn.GELU()
        )

        fusion_dim = vision_dim + emb_dim
        self.regression_head = nn.Sequential(
            nn.Linear(fusion_dim, 512),
            nn.BatchNorm1d(512),
            nn.GELU(),
            nn.Dropout(p=dropout),
            nn.Linear(512, 128),
            nn.BatchNorm1d(128),
            nn.GELU(),
            nn.Dropout(p=dropout * 0.5),
            nn.Linear(128, 1),
            nn.ReLU()
        )

    def extract_visual_features(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x)
        return torch.flatten(x, 1)

    def forward(self, image, treatment_one_hot):
        vis = self.extract_visual_features(image)
        treat = self.treatment_net(treatment_one_hot)
        fused = torch.cat([vis, treat], dim=1)
        return self.regression_head(fused)

def freeze_backbone(model):
    if hasattr(model, 'features'):
        for p in model.features.parameters():
            p.requires_grad = False
    elif hasattr(model, 'layer1'):
        for l in [model.conv1, model.bn1, model.layer1, model.layer2, model.layer3, model.layer4]:
            for p in l.parameters():
                p.requires_grad = False

def unfreeze_backbone(model):
    for p in model.parameters():
        p.requires_grad = True

def get_optimizer_groups(model, backbone_lr=1e-5, head_lr=1e-3, weight_decay=1e-4):
    head_names = {'treatment_net', 'regression_head'}
    backbone_params, head_params = [], []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if any(h in name for h in head_names):
            head_params.append(p)
        else:
            backbone_params.append(p)
    return [
        {'params': backbone_params, 'lr': backbone_lr, 'weight_decay': weight_decay},
        {'params': head_params, 'lr': head_lr, 'weight_decay': weight_decay}
    ]

print('Multimodal RSL architectures with Part I weight transfer initialized successfully.')


### 4. Training Engine with Huber Loss (Smooth L1) & Early Stopping
- **Loss:** Smooth L1 (Huber Loss, $\beta=1.0$) for robust error penalization.
- **Stage 1 (Warmup):** 5 epochs on fusion head with frozen backbone.
- **Stage 2 (Fine-Tuning):** 30 epochs end-to-end with Cosine Annealing learning rate schedule.


In [ ]:
def compute_metrics(y_true, y_pred):
    mae = float(np.mean(np.abs(y_true - y_pred)))
    rmse = float(np.sqrt(np.mean((y_true - y_pred) ** 2)))
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    ss_res = np.sum((y_true - y_pred) ** 2)
    r2 = float(1.0 - (ss_res / (ss_tot + 1e-8)))
    return {'mae': mae, 'rmse': rmse, 'r2': r2}

def train_rsl_model(model_name, total_epochs=35, warmup_epochs=5, patience=8):
    print('=' * 75)
    print(f'STARTING MULTIMODAL RSL TRAINING: {model_name.upper()}')
    print('=' * 75)

    model = (MultimodalResNetRSL() if model_name == 'resnet50' else MultimodalSwinRSL()).to(device)
    criterion = nn.SmoothL1Loss(beta=1.0)
    os.makedirs('checkpoints_rsl', exist_ok=True)
    best_ckpt = f'checkpoints_rsl/best_{model_name}_rsl.pth'

    history = {'train_loss': [], 'val_loss': [], 'train_mae': [], 'val_mae': [], 'val_rmse': [], 'val_r2': []}

    # Stage 1: Warmup
    if warmup_epochs > 0:
        print(f'>>> Stage 1: Fusion Head Warmup ({warmup_epochs} epochs, frozen backbone) <<<')
        freeze_backbone(model)
        optimizer_warmup = AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3, weight_decay=1e-4)

        for ep in range(1, warmup_epochs + 1):
            model.train()
            t_loss, p_list, t_list, n_tot = 0.0, [], [], 0
            for imgs, treats, targets in train_loader:
                imgs, treats, targets = imgs.to(device), treats.to(device), targets.to(device)
                optimizer_warmup.zero_grad()
                outs = model(imgs, treats)
                loss = criterion(outs, targets)
                loss.backward()
                optimizer_warmup.step()

                bs = targets.size(0)
                t_loss += loss.item() * bs
                n_tot += bs
                p_list.append(outs.detach().cpu().numpy())
                t_list.append(targets.detach().cpu().numpy())

            tr_mae = compute_metrics(np.vstack(t_list).flatten(), np.vstack(p_list).flatten())['mae']

            model.eval()
            v_loss, vp_list, vt_list, vn_tot = 0.0, [], [], 0
            with torch.no_grad():
                for imgs, treats, targets in val_loader:
                    imgs, treats, targets = imgs.to(device), treats.to(device), targets.to(device)
                    outs = model(imgs, treats)
                    bs = targets.size(0)
                    v_loss += criterion(outs, targets).item() * bs
                    vn_tot += bs
                    vp_list.append(outs.cpu().numpy())
                    vt_list.append(targets.cpu().numpy())

            v_met = compute_metrics(np.vstack(vt_list).flatten(), np.vstack(vp_list).flatten())
            print(f'  [Warmup {ep:02d}/{warmup_epochs:02d}] Train Loss: {t_loss/n_tot:.4f}, MAE: {tr_mae:.2f}d || Val Loss: {v_loss/vn_tot:.4f}, MAE: {v_met["mae"]:.2f}d, R²: {v_met["r2"]:.3f}')

    # Stage 2: Fine-Tuning
    print(f'>>> Stage 2: Full Fine-Tuning ({total_epochs - warmup_epochs} epochs) <<<')
    unfreeze_backbone(model)
    optimizer = AdamW(get_optimizer_groups(model, backbone_lr=1e-5, head_lr=5e-4))
    scheduler = CosineAnnealingLR(optimizer, T_max=total_epochs - warmup_epochs, eta_min=1e-7)

    best_val_mae = float('inf')
    patience_cnt = 0

    for ep in range(1, total_epochs - warmup_epochs + 1):
        cur_ep = warmup_epochs + ep
        model.train()
        t_loss, p_list, t_list, n_tot = 0.0, [], [], 0
        for imgs, treats, targets in train_loader:
            imgs, treats, targets = imgs.to(device), treats.to(device), targets.to(device)
            optimizer.zero_grad()
            outs = model(imgs, treats)
            loss = criterion(outs, targets)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            bs = targets.size(0)
            t_loss += loss.item() * bs
            n_tot += bs
            p_list.append(outs.detach().cpu().numpy())
            t_list.append(targets.detach().cpu().numpy())
        scheduler.step()

        tr_mae = compute_metrics(np.vstack(t_list).flatten(), np.vstack(p_list).flatten())['mae']

        model.eval()
        v_loss, vp_list, vt_list, vn_tot = 0.0, [], [], 0
        with torch.no_grad():
            for imgs, treats, targets in val_loader:
                imgs, treats, targets = imgs.to(device), treats.to(device), targets.to(device)
                outs = model(imgs, treats)
                bs = targets.size(0)
                v_loss += criterion(outs, targets).item() * bs
                vn_tot += bs
                vp_list.append(outs.cpu().numpy())
                vt_list.append(targets.cpu().numpy())

        v_met = compute_metrics(np.vstack(vt_list).flatten(), np.vstack(vp_list).flatten())
        va_l = v_loss / vn_tot

        history['train_loss'].append(t_loss / n_tot)
        history['val_loss'].append(va_l)
        history['train_mae'].append(tr_mae)
        history['val_mae'].append(v_met['mae'])
        history['val_rmse'].append(v_met['rmse'])
        history['val_r2'].append(v_met['r2'])

        mark = ''
        if v_met['mae'] < best_val_mae:
            best_val_mae = v_met['mae']
            patience_cnt = 0
            torch.save({'state_dict': model.state_dict(), 'val_mae': best_val_mae, 'metrics': v_met}, best_ckpt)
            mark = ' [* BEST]'
        else:
            patience_cnt += 1

        print(f'  [Epoch {cur_ep:02d}/{total_epochs:02d}] Train Loss: {t_loss/n_tot:.4f}, MAE: {tr_mae:.2f}d || Val Loss: {va_l:.4f}, MAE: {v_met["mae"]:.2f}d, R²: {v_met["r2"]:.3f}{mark}')
        if patience_cnt >= patience:
            print(f'Early stopping triggered after {patience} epochs without improvement.')
            break

    print(f'Finished {model_name}. Best Val MAE: {best_val_mae:.2f} days | Checkpoint: {best_ckpt}')
    return history


### 5. Training Execution: Multimodal ResNet50 & Swin Transformer


In [ ]:
# Train Multimodal ResNet50
hist_resnet = train_rsl_model('resnet50', total_epochs=35, warmup_epochs=5, patience=8)

# Train Multimodal Swin-T
hist_swin = train_rsl_model('swin_t', total_epochs=35, warmup_epochs=5, patience=8)

# Plot Comparative Regression Learning Curves
os.makedirs('outputs_rsl', exist_ok=True)
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.plot(hist_resnet['val_loss'], 'b-o', label='ResNet50 Val Loss')
plt.plot(hist_swin['val_loss'], 'r-s', label='Swin-T Val Loss')
plt.title('Validation Smooth L1 Loss', fontweight='bold')
plt.xlabel('Fine-Tuning Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True, linestyle='--')

plt.subplot(1, 2, 2)
plt.plot(hist_resnet['val_mae'], 'b-o', label='ResNet50 Val MAE (Days)')
plt.plot(hist_swin['val_mae'], 'r-s', label='Swin-T Val MAE (Days)')
plt.title('Validation Mean Absolute Error (MAE in Days)', fontweight='bold')
plt.xlabel('Fine-Tuning Epoch')
plt.ylabel('MAE (Days)')
plt.legend()
plt.grid(True, linestyle='--')

plt.tight_layout()
plt.savefig('outputs_rsl/rsl_training_curves_comparison.png', dpi=200)
plt.show()


### 6. Held-Out Test Set Evaluation & Actual vs. Predicted Plots
Evaluates unseen physical mushroom samples (IDs: 1, 4, 21) across all storage days.


In [ ]:
def evaluate_test_rsl(model_name):
    model = (MultimodalResNetRSL(pretrained=False) if model_name == 'resnet50' else MultimodalSwinRSL(pretrained=False)).to(device)
    ckpt = torch.load(f'checkpoints_rsl/best_{model_name}_rsl.pth', map_location=device)
    model.load_state_dict(ckpt['state_dict'])
    model.eval()

    y_true, y_pred = [], []
    with torch.no_grad():
        for imgs, treats, targets in test_loader:
            imgs, treats = imgs.to(device), treats.to(device)
            outs = model(imgs, treats)
            y_true.extend(targets.squeeze().cpu().numpy())
            y_pred.extend(outs.squeeze().cpu().numpy())

    y_true, y_pred = np.array(y_true), np.array(y_pred)
    metrics = compute_metrics(y_true, y_pred)

    print('=' * 60)
    print(f'{model_name.upper()} FINAL TEST SET PERFORMANCE:')
    print(f'  - Mean Absolute Error (MAE):  {metrics["mae"]:.2f} days')
    print(f'  - Root Mean Squared Err (RMSE): {metrics["rmse"]:.2f} days')
    print(f'  - Coefficient of Det (R²):     {metrics["r2"]:.4f}')
    print('=' * 60)

    # Plot Scatter Actual vs. Predicted
    plt.figure(figsize=(7, 6))
    plt.scatter(y_true, y_pred, color='#2ca02c' if model_name == 'swin_t' else '#1f77b4',
                alpha=0.7, edgecolors='k', s=65, label='Test Samples (IDs: 1, 4, 21)')
    max_val = max(float(y_true.max()), float(y_pred.max()), 20.0)
    plt.plot([0, max_val], [0, max_val], 'r--', linewidth=2, label='Ideal Fit (y = x)')
    plt.xlabel('Actual Remaining Shelf Life (Days)', fontweight='bold', fontsize=11)
    plt.ylabel('Predicted Remaining Shelf Life (Days)', fontweight='bold', fontsize=11)
    plt.title(f'{model_name.upper()} RSL Prediction on Test Set\nMAE: {metrics["mae"]:.2f}d | RMSE: {metrics["rmse"]:.2f}d | R²: {metrics["r2"]:.3f}', fontweight='bold')
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.legend()
    plt.tight_layout()
    plt.savefig(f'outputs_rsl/{model_name}_test_actual_vs_pred.png', dpi=200)
    plt.show()

evaluate_test_rsl('resnet50')
evaluate_test_rsl('swin_t')


### 7. Qualitative Sample Inference & Comparison across All 3 Treatments
Extracts representative physical test samples (IDs: 1, 4, 21) across Control, 1SKEO.2CEO, and 2SKEO.1CEO,
comparing Actual RSL vs. ResNet50, Swin-T, and Ensemble predictions.

In [ ]:
# Load best trained models
res_model = MultimodalResNetRSL(pretrained=False).to(device)
res_ckpt = torch.load("checkpoints_rsl/best_resnet50_rsl.pth", map_location=device)
res_model.load_state_dict(res_ckpt["state_dict"])
res_model.eval()

swin_model = MultimodalSwinRSL(pretrained=False).to(device)
swin_ckpt = torch.load("checkpoints_rsl/best_swin_t_rsl.pth", map_location=device)
swin_model.load_state_dict(swin_ckpt["state_dict"])
swin_model.eval()

# Select diverse representative test samples across all 3 treatments and storage days
candidate_keys = [
    ("Control", 0),
    ("Control", 3),
    ("Control", 7),
    ("1SKEO.2CEO", 0),
    ("1SKEO.2CEO", 7),
    ("1SKEO.2CEO", 11),
    ("1SKEO.2CEO", 20),
    ("2SKEO.1CEO", 0),
    ("2SKEO.1CEO", 7),
    ("2SKEO.1CEO", 11),
    ("2SKEO.1CEO", 20),
]

selected_samples = []
seen = set()
for idx in range(len(test_ds)):
    meta = test_ds.samples[idx]
    key = (meta["treatment"], meta["day"])
    if key in candidate_keys and key not in seen:
        seen.add(key)
        selected_samples.append((idx, meta))
    if len(selected_samples) >= 9:
        break

print("=" * 105)
print(f"{'Treatment':<14} | {'Day':<4} | {'ID':<3} | {'Actual RSL':<11} | {'ResNet Pred':<12} | {'ResNet Err':<11} | {'Swin Pred':<10} | {'Swin Err':<9} | {'Ensemble':<9}")
print("=" * 105)

plot_data = []
with torch.no_grad():
    for ds_idx, meta in selected_samples:
        img, treat, target = test_ds[ds_idx]
        img_in = img.unsqueeze(0).to(device)
        treat_in = treat.unsqueeze(0).to(device)

        p_res = res_model(img_in, treat_in).item()
        p_swin = swin_model(img_in, treat_in).item()
        p_ens = 0.6 * p_res + 0.4 * p_swin  # Weighted ensemble
        actual = target.item()

        err_res = abs(p_res - actual)
        err_swin = abs(p_swin - actual)

        print(f"{meta['treatment']:<14} | Day {meta['day']:<2} | #{meta['sample_id']:<2} | {actual:5.1f} days  | {p_res:5.2f} days  | {err_res:5.2f} days  | {p_swin:5.2f} days | {err_swin:5.2f} days| {p_ens:5.2f} days")
        plot_data.append((meta, img, actual, p_res, p_swin, p_ens))
print("=" * 105)

# Plot Visual Grid with Predictions
n_show = min(len(plot_data), 6)
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.flatten()
mean = np.array([0.485, 0.456, 0.406])
std = np.array([0.229, 0.224, 0.225])

for i in range(n_show):
    meta, img_tensor, act, pr, ps, pe = plot_data[i]
    # Un-normalize image for plotting
    img_np = img_tensor.permute(1, 2, 0).cpu().numpy()
    img_np = np.clip(img_np * std + mean, 0, 1)

    axes[i].imshow(img_np)
    axes[i].set_title(f"{meta['treatment']} - Day {meta['day']} (ID #{meta['sample_id']})
"
                     f"Actual RSL: {act:.1f}d
"
                     f"ResNet: {pr:.1f}d | Swin: {ps:.1f}d | Ens: {pe:.1f}d",
                     fontsize=10, fontweight="bold",
                     color="darkgreen" if abs(pe - act) <= 1.5 else "darkred")
    axes[i].axis("off")

plt.tight_layout()
plt.savefig("outputs_rsl/test_samples_predictions.png", dpi=200)
plt.show()
print("Inference visualization saved to outputs_rsl/test_samples_predictions.png")
